# I.P.$^+$ for the VGGs

Gives VGG-11 and VGG-19 the same budget-matched baseline the ResNets are
getting: 50 pruning epochs plus the 25-epoch mask-frozen fine-tune, at I.P.'s
own SGD 0.01.

**These resume rather than rerun.** Every VGG I.P. cell already saved its sparse
checkpoint, so `--finetune_only` loads it, reads the mask back off the zeros,
and runs only the 25 fine-tune epochs. That is equivalent to running 50+25 in
one process -- `finetune()` builds a fresh AdamW optimizer either way, so no
optimizer state is lost -- and costs a third of the time.

| | cells | hours |
|---|---:|---:|
| rerun 50+25 | 72 | 21.2 |
| resume + 25 | 72 | **~7.3** |

Sparsities 0.95, 0.97, 0.99, 0.999; magnitude, snip, wanda; seeds 1-3. Records
carry the `.ft25` suffix, matching the ResNet ablation.

The ResNet `.ft25` cells ran end-to-end while these resume, so the appendix
should say so.


In [ ]:
import sys, pathlib

here = pathlib.Path.cwd()
while not (here / '.git').exists() and here != here.parent:
    here = here.parent
sys.path.insert(0, str(here / 'project' / 'test_notebooks'))

import nb_common as nb
info = nb.setup()


## Plan

Resolves each cell's sparse checkpoint, then guards that the fine-tune is the only deviation.

In [ ]:
import json, glob, os, re

MODELS     = ('vgg11', 'vgg19')
PRUNERS    = ('magnitude', 'snip', 'wanda')
SPARSITIES = (0.95, 0.97, 0.99, 0.999)
SEEDS      = (1, 2, 3)
GPU        = 0

FT = dict(enable_finetune=True, epochs_ft=25,
          optimizer_type_ft='adamw', learning_rate_ft=1e-4,
          finetune_only=True)

# Where each I.P. run left its sparse weights. Sparsity carries a dot, so the
# key is matched with an anchored regex; splitting on '.' tears 0.95 apart.
KEY = re.compile(r'^static\.prune\.(vgg11|vgg19)\.cifar10\.'
                 r's(0\.\d+)\.(magnitude|snip|wanda)\.seed(\d+)$')

root = os.environ['BACP_RESULTS_DIR']
sparse = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    m = KEY.match(r.get('experiment_group') or '')
    if m and r.get('status') == 'ok' and r.get('save_path'):
        sparse[(m.group(1), m.group(2), m.group(3), int(m.group(4)))] = r['save_path']

print('sparse checkpoints found: %d' % len(sparse))

plan, absent = [], []
for mdl in MODELS:
    for p in PRUNERS:
        for sp in SPARSITIES:
            for s in SEEDS:
                ckpt = sparse.get((mdl, str(sp), p, s))
                if ckpt is None or not os.path.exists(ckpt):
                    absent.append((mdl, sp, p, s, ckpt or '(no record)'))
                    continue
                plan.append(nb.make_cell(mdl, 'prune', seed=s, pruner=p, sparsity=sp,
                                         variant='ft25', trained_weights=ckpt, **FT))

if absent:
    print('\nMISSING %d checkpoint(s) -- these cells are skipped:' % len(absent))
    for a in absent:
        print('   ', a)

# guards: the fine-tune must be the only thing that differs from the I.P. arm
ref_keys = ('learning_rate', 'epochs', 'delta_T', 'sparsity_scheduler',
            'recovery_epochs', 'val_split', 'prune_task_head', 'wanda_group',
            'optimizer_type', 'batch_size', 'num_classes', 'dataset_name')
for c in plan:
    ref = nb.FAMILIES[c['model_name']]['prune']
    for k in ref_keys:
        if k in ref:
            assert c['config'][k] == ref[k], (c['key'], k, c['config'][k], ref[k])
    assert c['config']['learning_rate'] == 0.01, c['config']['learning_rate']
    assert c['config']['epochs'] == 50, 'epochs stays 50 so the record matches the ResNets'
    assert c['config']['finetune_only'] is True and c['config']['epochs_ft'] == 25
    assert c['key'].endswith('.ft25'), c['key']
    assert c['config']['trained_weights'].endswith('.pt')
    assert 'static-prune' in c['config']['trained_weights'], (
        'must resume a PRUNING checkpoint, not a dense or bacp one: '
        + c['config']['trained_weights'])

assert len({c['key'] for c in plan}) == len(plan), 'duplicate key'
print('\n%d cells, ~%.1f h (25 epochs each, resumed)'
      % (len(plan), len(plan) * 6.05 / 60))
assert nb.sanity_check(plan), 'sanity check failed'


## Run

In [ ]:
nb.run_group(plan, gpu=GPU)


## Results

In [ ]:
import json, glob, os, statistics as st

root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok' and '.smoke' not in k:
        acc[k] = r.get('test_acc_exact_pct') or r.get('test_acc_pct')

def cell(arm, m, sp, p, suffix=''):
    xs = [acc.get('static.%s.%s.cifar10.s%s.%s.seed%d%s' % (arm, m, sp, p, s, suffix))
          for s in SEEDS]
    xs = [x for x in xs if x is not None]
    if not xs:
        return None
    return st.mean(xs), (st.stdev(xs) if len(xs) > 1 else 0.0), len(xs)

def fmt(v):
    return '   --  ' if v is None else '%6.2f%s' % (v[0], '*' if v[2] < 3 else ' ')

print('%-26s %8s %8s %8s %8s %8s' % ('cell', 'I.P.', 'I.P.+', 'BaCP', 'D', "D'"))
print('-' * 70)
for m in MODELS:
    for p in PRUNERS:
        for sp in SPARSITIES:
            ip, ipp, bp = (cell('prune', m, sp, p), cell('prune', m, sp, p, '.ft25'),
                           cell('bacp', m, sp, p))
            d  = '%+7.2f' % (bp[0] - ip[0]) if ip and bp else '   --  '
            dp = '%+7.2f' % (bp[0] - ipp[0]) if ipp and bp else '   --  '
            print('%-26s %8s %8s %8s %8s %8s'
                  % ('%s %s %s' % (m, p, sp), fmt(ip), fmt(ipp), fmt(bp), d, dp))
print()
print("* = fewer than 3 seeds.  D' is against the budget-matched control.")
